# Sentiment Feature Generation 
Loads `Processed_Reviews.csv`, predicts sentiment for combined `Title + Text` using `cardiffnlp/twitter-roberta-base-sentiment`, and appends only `Sentiment` as the final column.

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm.auto import tqdm

# Config
INPUT_FILE = "Processed_Reviews.csv"
OUTPUT_FILE = INPUT_FILE
BATCH_SIZE = 32
MAX_TEXT_LEN = 256
PIPELINE_DEVICE = 0 if torch.cuda.is_available() else -1

# Load dataset
df = pd.read_csv(INPUT_FILE)

# Build combined text from Title + Text
title = df["Title"].fillna("").astype(str) if "Title" in df.columns else pd.Series([""] * len(df))
text = df["Text"].fillna("").astype(str) if "Text" in df.columns else pd.Series([""] * len(df))
combined_text = (title + " " + text).str.strip()

# Load only the requested sentiment model
sentiment_pipe = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    device=PIPELINE_DEVICE,
)

# Predict sentiment in batches
label_map = {"LABEL_0": "NEGATIVE", "LABEL_1": "NEUTRAL", "LABEL_2": "POSITIVE"}
predicted = [None] * len(combined_text)
batch_starts = range(0, len(combined_text), BATCH_SIZE)

for start in tqdm(batch_starts, desc="Predicting Sentiment", unit="batch"):
    batch_texts = combined_text.iloc[start:start + BATCH_SIZE].tolist()
    non_empty = []
    for offset, t in enumerate(batch_texts):
        if isinstance(t, str) and t.strip():
            non_empty.append((offset, t))
    if not non_empty:
        continue

    offsets = [offset for offset, _ in non_empty]
    texts = [t for _, t in non_empty]
    outputs = sentiment_pipe(texts, truncation=True, max_length=MAX_TEXT_LEN, batch_size=BATCH_SIZE)

    for offset, out in zip(offsets, outputs):
        raw_label = str(out.get("label", "")).strip().upper()
        predicted[start + offset] = label_map.get(raw_label, raw_label if raw_label else None)

# Append only this new column at the end
df["Sentiment"] = predicted

# Save
df.to_csv(OUTPUT_FILE, index=False)
print("Saved:", OUTPUT_FILE)
print("Final shape:", df.shape)
print(df[["Sentiment"]].head())